# Cross-Free Families and Symbolica

FeynKit turns a loop diagram into a typed Cross-Free Family (CFF): accepted acyclic orientations, their denominator products, and the energy or hybrid surfaces referenced by those products. The final expression lives in the same Symbolica kernel as the rest of the package.

In [ ]:
from pathlib import Path

from IPython.display import display
from symbolica import S
import symbolica.community.feynkit as fk

DATA = next(path for path in (Path("data"), Path("examples/feynkit/data")) if path.exists())
model = fk.Model(DATA / "scalars_2p_3p.json")
options = fk.GenerationOptions(max_vertices=3, allow_self_loops=True)
options.add_vertex_allow(["V_3_SCALAR_000"])
generated = model.generate_diagrams(
    ["scalar_0"],
    [1000, "scalar_0"],
    loops=(0, 1),
    options=options,
)
diagram = next(
    item for item in generated.diagrams
    if item.loop_count == 1 and all(edge.source != edge.target for edge in item.edges)
)
diagram.validate(model)
(diagram.name, diagram.loop_count, len(diagram.edges))

## Edge IDs are the control surface

CFF orientation constraints, contractions, and initial-state declarations all refer to the stable edge IDs shown below.

In [ ]:
[
    (edge.id, edge.source, edge.target, edge.particle_name)
    for edge in diagram.edges
]

## Generate the CFF

The diagram owns this conversion. For large graphs, set `max_orientations` as an explicit combinatorial guard.

In [ ]:
cff = diagram.build_cff(max_orientations=10_000)
report = cff.report
{
    "candidate_orientations": report.candidate_orientations,
    "acyclic_orientations": report.acyclic_orientations,
    "unfolded_terms": report.unfolded_terms,
    "interned_surfaces": report.interned_surfaces,
}

## Inspect surfaces and denominator products

Energy surfaces contain positive on-shell energies and an external shift. Hybrid (H) surfaces may also contain negative energies. Special unit/infinite surfaces have no Symbolica symbol.

In [ ]:
surface_rows = [
    {
        "symbol": surface.symbol_name,
        "kind": surface.kind,
        "positive_energies": surface.positive_energies,
        "negative_energies": surface.negative_energies,
        "external_shift": surface.external_shift,
        "vertices": surface.vertices,
    }
    for surface in cff.surfaces
]
surface_rows

In [ ]:
orientation = cff.orientations[0]
{
    "orientation_id": orientation.id,
    "edge_orientations": orientation.edge_orientations,
    "denominator_products": [
        [surface.symbol_name or surface.kind for surface in product]
        for product in orientation.denominator_products()
    ],
}

## Continue symbolically

`to_expression()` maps CFF surface IDs to names such as `feynkit::E0` and `feynkit::H0`. Their typed definitions remain available in `cff.surfaces`, so symbolic algebra and physics metadata stay connected.

In [ ]:
expression = cff.to_expression()
weighted = S("g")**2 * expression
display(expression.formatted())
display(weighted.formatted())

## Advanced controls

Use `diagram.build_cff(fixed_orientations={edge_id: reversed}, contracted_edges=[...], initial_state_edges=[...])` for constrained constructions. Here `False` keeps an edge's stored direction and `True` reverses it. Apply constraints only after inspecting the diagram's edge table.